# Bronze Layer Ingestion — Olist Pipeline

**Goal of this notebook:** take the 9 raw Olist CSV files that Azure Data Factory
landed in ADLS Gen2, and register them as Bronze tables in Unity Catalog.

**Bronze rule:** load the data exactly as it arrived. No cleaning, no renaming,
no type conversions. Bronze is a faithful copy of the source, so that if a
question comes up later, there is always an untouched version to go back to.
Cleaning happens in Silver.

**How:** Databricks Auto Loader, which keeps track of which files it has already
read. That means re-running this notebook does not reload the same data twice.

**Destination:** `olist_catalog.bronze`

## Step 1 — Confirm Databricks can see the raw files

Before writing any ingestion code, check that Databricks can actually reach the
storage folder and see all 9 CSV files.

**This is deliberate:** if something is wrong with permissions or the path, it is
much easier to spot here than buried inside a failed ingestion run later.

In [0]:
# The ADLS storage account and container I set up in Part 1.
# Kept as variables so the long address below is only written once,
# and so it is easy to change in one place if anything moves.
storage_account = "oluwasaanumi"
container       = "olist"

# Build the full address of the raw folder.
# abfss:// is the protocol Azure Data Lake Storage Gen2 uses.
# This is the same address ADF wrote the 9 CSVs into.
raw_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/data_zone/raw"

# dbutils.fs.ls means "list the files in this folder".
# It returns one entry per file, each with a name and a size in bytes.
for f in dbutils.fs.ls(raw_path):
    # f.size is in bytes, so divide twice by 1024 to get megabytes.
    # The :<45 and :>8.2f just line the columns up neatly.
    print(f"{f.name:<45} {f.size/1024/1024:>8.2f} MB")

## Step 2 — Look inside one file

Before automating the load of all 9 files, read just one and look at it.

The point is to confirm two things: the CSV has a proper header row, and the
columns are what we expect. Doing this on one file now is cheaper than
discovering a surprise halfway through loading all nine.

In [0]:
# Read a single CSV into a DataFrame.
# A DataFrame is just a table held in memory - nothing is saved anywhere yet.
sample_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(f"{raw_path}/olist_customers_dataset.csv")
)

# display() shows the table as a scrollable grid
# .limit(5) means only pulls 5 rows
display(sample_df.limit(5))

In [0]:
# printSchema() lists every colum and the data type Spark assigned to it.
sample_df.printSchema()

## Step 3 — Create the medallion schemas
**What a schema is:** a catalog holds schemas, and a schema holds tables.
Roughly: catalog is the building, schema is the floor, table is the room.
Nothing can be written until the floor exists.

**What we are doing:** creating one schema per medallion layer, so each stage of
the pipeline has a clear home — raw data in `bronze`, cleaned data in `silver`,
aggregated data in `gold`.

All three are created now rather than one at a time, since the structure is
already decided and there is no reason to come back for the other two later.

Each is created with a comment describing what belongs in it, so the purpose of
each layer is visible to anyone browsing the catalog.

In [0]:
%sql
-- A schema is a folder inside a catalog that groups related tables.
-- We create one per medallion layer, so each stage of the pipeline has its own home.
-- IF NOT EXISTS means: skip quietly if it is already there, rather than error.
-- The COMMENT is a note stored with the schema, so anyone browsing the catalog
-- later can see what belongs in it without having to ask.

CREATE SCHEMA IF NOT EXISTS olist_catalog.bronze
  COMMENT 'Raw Olist data exactly as ingested from ADLS. No cleaning or type changes.';

CREATE SCHEMA IF NOT EXISTS olist_catalog.silver
  COMMENT 'Cleaned, typed and deduplicated Olist data.';

CREATE SCHEMA IF NOT EXISTS olist_catalog.gold
  COMMENT 'Aggregated, business-ready tables for reporting and Power BI.';

In [0]:
%sql
SHOW SCHEMAS IN olist_catalog;

## Step 4 — Load one table with Auto Loader

Auto Loader is Databricks' tool for loading files that keep arriving over time.
Its useful trick: it remembers which files it has already read, by keeping notes
in a folder called a checkpoint. Run it again and it only picks up what is new.

That is why `checkpoint_zone` was created back in Part 1, separate from the data.
Checkpoints are pipeline bookkeeping, not data.

Starting with one table only. If something is wrong, it is far easier to see here
than across nine loads at once.

In [0]:
from pyspark.sql import functions as F

# Where Auto Loader keeps its notes about what it has already read.
# This is checkpoint_zone from Part 1 — separate from the data itself.
checkpoint_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net/checkpoint_zone/bronze"

# Doing customers first, on its own.
table_name  = "customers"
source_file = "olist_customers_dataset.csv"

# readStream means "keep watching this folder", rather than "read it once".
bronze_df = (
    spark.readStream
        .format("cloudFiles")                        # cloudFiles is Auto Loader
        .option("cloudFiles.format", "csv")          # the incoming files are CSVs
        .option("cloudFiles.schemaLocation",         # where Auto Loader stores the
                f"{checkpoint_root}/{table_name}/schema")   # column names it worked out
        .option("header", "true")                    # first line is column names
        .option("pathGlobFilter", source_file)       # ignore everything except this one file
        .load(raw_path)                              # the folder to watch

        # Add two tracking columns of our own.
        # Prefixed with _ so they are visibly ours, not part of the source data.
        .select(
            "*",                                                  # keep every original column
            F.col("_metadata.file_name").alias("_source_file"),   # which file this row came from
            F.current_timestamp().alias("_ingested_at"),          # when we loaded it
        )
)

# writeStream sends the result somewhere. Here: a table in Unity Catalog.
query = (
    bronze_df.writeStream
        .option("checkpointLocation",                # the notes on what has been processed
                f"{checkpoint_root}/{table_name}/checkpoint")
        .trigger(availableNow=True)                  # process what is here now, then stop
        .toTable(f"olist_catalog.bronze.{table_name}")   # create/append to this table
)

# Wait here until the load finishes, so the next cell does not run too early.
query.awaitTermination()

print(f"Done: olist_catalog.bronze.{table_name}")

### Check the customers table before loading the rest

Three things worth confirming on one table, before repeating the pattern nine times:

1. The row count matches the source file.
2. `_rescued_data` is empty, meaning nothing was malformed or unexpected.
3. The tracking columns were populated correctly.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    -- Count rows where Auto Loader had to rescue something. Should be 0
    COUNT(_rescued_data) AS rows_with_rescued_data,
    -- Confirm every row knows which file it came from.
    COUNT(DISTINCT _source_file) AS distinct_source_files,
    MIN(_ingested_at) AS first_loaded_at
FROM olist_catalog.bronze.customers

## Step 5 — Load all nine files

The single-table load in Step 4 worked, so the same pattern now applies to all
nine files.

**The approach:** rather than copying the Step 4 block nine times, the file list
is written once as configuration, and one function does the loading. Adding a
tenth file later means adding one line to a list, not pasting another block.

**Table naming:** the `olist_` prefix and `_dataset` suffix are dropped, since
the catalog and schema already say `olist_catalog.bronze`. The original filename
is still recorded in the `_source_file` column of every row.

**One special case:** `olist_order_reviews_dataset.csv` contains customer review
text, which includes line breaks and commas inside quoted fields. Read with the
default settings, a single review spanning two lines would be read as two broken
rows. The `multiLine` option tells Spark to respect the quotes instead of
treating every line break as a new row.

**Note on re-running:** customers was already loaded in Step 4. Because Auto
Loader keeps a checkpoint per table, running it again loads nothing new rather
than duplicating it.

In [0]:
# One entry per file. This is the only place the list lives.
# Adding a new source later means adding one line here, nothing else.
bronze_tables = [
    {"table": "customers",       "file": "olist_customers_dataset.csv"},
    {"table": "geolocation",     "file": "olist_geolocation_dataset.csv"},
    {"table": "order_items",     "file": "olist_order_items_dataset.csv"},
    {"table": "order_payments",  "file": "olist_order_payments_dataset.csv"},
    {"table": "orders",          "file": "olist_orders_dataset.csv"},
    {"table": "products",        "file": "olist_products_dataset.csv"},
    {"table": "sellers",         "file": "olist_sellers_dataset.csv"},
    {"table": "product_category_translation",
                                 "file": "product_category_name_translation.csv"},

    # Review text contains line breaks and commas inside quoted fields.
    # multiLine tells Spark to respect the quotes rather than breaking on every newline.
    {"table": "order_reviews",   "file": "olist_order_reviews_dataset.csv",
     "extra_options": {"multiLine": "true", "escape": '"'}},
]

In [0]:
def load_bronze_table(table, file, extra_options=None):
    """Load one CSV from raw into a Bronze table using Auto Loader.

    This is the exact same logic as Step 4, with the table name and
    filename passed in instead of hard-coded.
    """

    # Build the reader. Same options as Step 4.
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{checkpoint_root}/{table}/schema")
            .option("header", "true")
            .option("pathGlobFilter", file)
    )

    # Apply any file-specific options, such as multiLine for order_reviews.
    # Files without extra options simply skip this loop.
    for key, value in (extra_options or {}).items():
        reader = reader.option(key, value)

    # Add the same two tracking columns to every table.
    df = reader.load(raw_path).select(
        "*",
        F.col("_metadata.file_name").alias("_source_file"),
        F.current_timestamp().alias("_ingested_at"),
    )

    # Write to the table, then wait for it to finish before returning.
    query = (
        df.writeStream
            .option("checkpointLocation", f"{checkpoint_root}/{table}/checkpoint")
            .trigger(availableNow=True)
            .toTable(f"olist_catalog.bronze.{table}")
    )
    query.awaitTermination()

In [0]:
# Load each table in turn, printing progress as it goes.
for cfg in bronze_tables:
    print(f"Loading {cfg['table']} ...")
    load_bronze_table(
        table=cfg["table"],
        file=cfg["file"],
        extra_options=cfg.get("extra_options"),   # None for most files
    )
    print(f"  done: olist_catalog.bronze.{cfg['table']}")

print("\nAll tables loaded.")

## Step 6 — Validate the Bronze layer

Loading without error is not the same as loading correctly. Three checks:

1. **Row counts** match the known row counts of the Olist dataset. This catches
   silently dropped or duplicated rows.
2. **`_rescued_data` is empty** on every table. Anything rescued means a row did
   not fit the expected shape — most likely on `order_reviews`, if the multiLine
   setting had not been applied.
3. **One source file per table**, confirming no file leaked into the wrong table
   through a mistaken filter.

In [0]:
# The published row counts for the Olist dataset, used as the expected values.
# If a count does not match, something went wrong in ingestion and needs investigating.
expected_counts = {
    "customers":                    99441,
    "geolocation":                1000163,
    "order_items":                 112650,
    "order_payments":              103886,
    "order_reviews":                99224,
    "orders":                       99441,
    "products":                     32951,
    "sellers":                       3095,
    "product_category_translation":    71,
}

results = []

for cfg in bronze_tables:
    table = cfg["table"]

    # Ask each table for its counts in a single pass.
    row = spark.sql(f"""
        SELECT
            COUNT(*)                     AS actual_rows,
            COUNT(_rescued_data)         AS rescued_rows,
            COUNT(DISTINCT _source_file) AS source_files
        FROM olist_catalog.bronze.{table}
    """).collect()[0]

    expected = expected_counts[table]

    results.append({
        "table":        table,
        "expected":     expected,
        "actual":       row["actual_rows"],
        # A simple pass/fail flag is easier to scan than comparing numbers by eye.
        "match":        "OK" if row["actual_rows"] == expected else "MISMATCH",
        "rescued":      row["rescued_rows"],
        "source_files": row["source_files"],
    })

# Show the results as a table.
display(spark.createDataFrame(results))

## Step 7 — Confirm re-running is safe

The reason for choosing Auto Loader over a plain batch read was that it keeps a
checkpoint of which files it has already processed, so re-running does not
reload the same data.

That is a claim worth testing rather than assuming. The whole load is run a
second time, and the row counts checked again. If the counts are unchanged, the
checkpoint is doing its job. If they have doubled, it is not.

## Where things stand

All 9 Olist CSVs are now registered as Bronze tables in `olist_catalog.bronze`,
loaded with Auto Loader, with row counts matching the source and no rescued data.

Every table carries two tracking columns: `_source_file` (which file the row came
from) and `_ingested_at` (when it was loaded).

Re-running the notebook was tested and loads nothing twice.

**Two things found along the way:**
- Reading files directly by path needs the **Storage Blob Delegator** role on the
  storage account, in addition to Storage Blob Data Contributor. The first role
  allows reading data; the second allows requesting the short-lived pass that
  path-based access uses.
- The `bronze`, `silver` and `gold` schemas did not exist in `olist_catalog` and
  had to be created before anything could be written.

**Next:** the Silver layer — typing the columns properly, handling nulls and
duplicates, and joining the tables into something usable.